# M3 — Modelo de Machine Learning y Recomendación
## Sistema de Recomendación Paralelo para E-Commerce — RetailRocket Dataset
### Entrega 2 — Implementación Inicial y Validación Parcial


| Criterio | Sección |
|---|---|
| Avance funcional | 2, 3, 4 (matriz de interacciones, split train/test, entrenamiento ALS) |
| Organización del código | Módulo `src/modelo_recomendacion.py` con clases y funciones documentadas |
| Validación inicial | 5 (Hit Rate@10, MAP@10, NDCG@10) |
| Gestión de problemas | 7 |
| Integración con M1/M2 | 1 (importa `pipeline_datos` y `analisis_eda` directamente) |

**Herramientas:** Implicit (ALS), SciPy (matrices dispersas), Polars.


## 0. Configuración inicial

In [1]:
# !pip install implicit scipy polars scikit-learn pyarrow


## 1. Importar los módulos de M1 y M2

En vez de repetir la ingesta o el clustering, importamos directamente las
funciones ya construidas y probadas en los módulos de los otros compañeros.


In [2]:
import sys
from pathlib import Path

# Ajusta esta ruta a la raíz de tu proyecto en VS Code
RAIZ_PROYECTO = Path(r"E:\Ing ciencia de datos\Septimo cuatri\Computacion paralela\clase 15\Codigos")
sys.path.append(str(RAIZ_PROYECTO))

from pipeline_datos import cargar_eventos_procesados          # Módulo M1
from analisis_eda import cargar_segmentos                     # Módulo M2
from modelo_recomendacion import (                             # Módulo M3
    construir_matriz_interacciones,
    dividir_train_test,
    ModeloALS,
    evaluar_modelo_batch,
)

CARPETA_PROCESSED = RAIZ_PROYECTO / "processed"
CARPETA_RESULTADOS = RAIZ_PROYECTO / "resultados"
CARPETA_RESULTADOS.mkdir(exist_ok=True)

print("Módulos de M1 y M2 importados correctamente.")


Módulos de M1 y M2 importados correctamente.


C:\Users\mora7\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Cargar los datos ya procesados por M1 y M2

No se repite la ingesta ni la limpieza: se usa directamente la salida de
`eventos_limpios.parquet` (M1) y `user_segments.parquet` (M2).


In [3]:
eventos = cargar_eventos_procesados(CARPETA_PROCESSED).collect()
segmentos = cargar_segmentos(CARPETA_PROCESSED)  # user_segments.parquet vive en processed/

print(f"Eventos cargados desde M1: {eventos.height:,} filas")
print(f"Segmentos cargados desde M2: {segmentos.height:,} usuarios, {segmentos['cluster'].n_unique()} clusters")


Eventos cargados desde M1: 2,755,641 filas
Segmentos cargados desde M2: 1,407,580 usuarios, 6 clusters


## 3. División Train/Test (leave-one-out temporal)

Para cada usuario con al menos 2 interacciones, se separa su interacción
**más reciente** como test y el resto como train. Esta es la estrategia
estándar para evaluar sistemas de recomendación con feedback implícito.


In [4]:
train, test = dividir_train_test(eventos, min_interacciones=2)


2026-07-26 12:09:15,944 [INFO] Dividiendo train/test (leave-one-out temporal)
2026-07-26 12:09:17,542 [INFO] Train: 2,349,652 filas | Test: 405,989 filas (405,989 usuarios evaluables)


## 4. Construcción de la matriz de interacciones y entrenamiento del modelo baseline (ALS)

Se usa `peso_implicito` (view=1, addtocart=3, transaction=5, definido en M1)
como valor de confianza. El modelo ALS se entrena en paralelo (multi-hilo)
sobre la matriz dispersa usuario-producto.


In [5]:
mi = construir_matriz_interacciones(train)

modelo = ModeloALS(factors=64, regularization=0.01, iterations=15)
tiempo_entrenamiento = modelo.entrenar(mi.matriz)

print(f"\nTiempo de entrenamiento: {tiempo_entrenamiento:.2f} segundos")
print(f"Dimensiones de la matriz: {mi.matriz.shape[0]:,} usuarios x {mi.matriz.shape[1]:,} productos")
print(f"Densidad: {100 * mi.matriz.nnz / (mi.matriz.shape[0] * mi.matriz.shape[1]):.4f}%")


2026-07-26 12:09:17,585 [INFO] Construyendo matriz de interacciones usuario-producto
2026-07-26 12:09:19,085 [INFO] Matriz construida: 1,407,580 usuarios x 227,081 productos (1,930,255 interacciones no nulas)
C:\Users\mora7\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
2026-07-26 12:09:19,478 [INFO] Entrenando ALS (factors=64, iterations=15) sobre 1407580 x 227081
100%|██████████| 15/15 [01:41<00:00,  6.78s/it]
2026-07-26 12:11:02,384 [INFO] Entrenamiento completado en 102.90 segundos



Tiempo de entrenamiento: 102.90 segundos
Dimensiones de la matriz: 1,407,580 usuarios x 227,081 productos
Densidad: 0.0006%


## 5. Validación inicial — Métricas de calidad (Hit Rate@10, MAP@10, NDCG@10)

Estas son las métricas de ranking que exige el Objetivo 2 del proyecto para
evaluar el modelo baseline, antes de compararlo con el modelo avanzado
(embeddings/Two-Tower) en la Entrega 3.


In [6]:
K = 10
metricas = evaluar_modelo_batch(modelo, mi.matriz, test, mi, k=K, max_usuarios=None, tam_lote=5000)

print(f"Usuarios evaluados: {metricas['usuarios_evaluados']:,}")
print(f"Hit Rate@{K}: {metricas[f'hit_rate@{K}']:.4f}")
print(f"MAP@{K}: {metricas[f'map@{K}']:.4f}")
print(f"NDCG@{K}: {metricas[f'ndcg@{K}']:.4f}")

import pandas as pd
pd.DataFrame([metricas]).to_csv(CARPETA_RESULTADOS / "metricas_modelo_baseline.csv", index=False)


2026-07-26 12:11:02,408 [INFO] Evaluando modelo (batch) con K=10
2026-07-26 12:11:08,307 [INFO] Evaluados 5,000 / 397,571 usuarios
2026-07-26 12:11:12,819 [INFO] Evaluados 10,000 / 397,571 usuarios
2026-07-26 12:11:17,181 [INFO] Evaluados 15,000 / 397,571 usuarios
2026-07-26 12:11:21,470 [INFO] Evaluados 20,000 / 397,571 usuarios
2026-07-26 12:11:25,364 [INFO] Evaluados 25,000 / 397,571 usuarios
2026-07-26 12:11:29,211 [INFO] Evaluados 30,000 / 397,571 usuarios
2026-07-26 12:11:33,334 [INFO] Evaluados 35,000 / 397,571 usuarios
2026-07-26 12:11:37,179 [INFO] Evaluados 40,000 / 397,571 usuarios
2026-07-26 12:11:40,770 [INFO] Evaluados 45,000 / 397,571 usuarios
2026-07-26 12:11:44,448 [INFO] Evaluados 50,000 / 397,571 usuarios
2026-07-26 12:11:48,520 [INFO] Evaluados 55,000 / 397,571 usuarios
2026-07-26 12:11:52,583 [INFO] Evaluados 60,000 / 397,571 usuarios
2026-07-26 12:11:56,624 [INFO] Evaluados 65,000 / 397,571 usuarios
2026-07-26 12:12:00,687 [INFO] Evaluados 70,000 / 397,571 usuario

Usuarios evaluados: 397,571
Hit Rate@10: 0.0266
MAP@10: 0.0124
NDCG@10: 0.0157


**Nota de interpretación:** `max_usuarios=2000` limita la evaluación a una
muestra por velocidad durante el desarrollo. Para el número final que va en
el informe, correr con `max_usuarios=None` (todos los usuarios evaluables)
sobre el dataset completo.


## 6. Integración con M2 — Rendimiento del modelo por segmento de usuario

Aquí es donde realmente se aprovecha la salida de M2: en vez de reportar una
sola métrica global, se compara qué tan bien recomienda el modelo para cada
uno de los 6 clusters de usuarios identificados por Ana María. Esto permite
detectar si el modelo funciona mejor para usuarios muy activos que para
usuarios ocasionales (Cluster 1, el 71% de la base).


In [7]:
import numpy as np
import polars as pl

filas_test = test.select(["visitorid", "itemid"]).to_numpy()
u_idx_arr = np.array([mi.visitorid_a_idx.get(int(v)) for v, _ in filas_test])
i_idx_arr = np.array([mi.itemid_a_idx.get(int(it)) for _, it in filas_test])
visitorid_arr = filas_test[:, 0].astype(int)

validos = (u_idx_arr != None) & (i_idx_arr != None)  # noqa: E711
u_idx_arr, i_idx_arr, visitorid_arr = u_idx_arr[validos].astype(int), i_idx_arr[validos].astype(int), visitorid_arr[validos]

resultados_por_usuario = []
TAM_LOTE = 5000
for inicio in range(0, len(u_idx_arr), TAM_LOTE):
    lote_u = u_idx_arr[inicio:inicio+TAM_LOTE]
    lote_i_real = i_idx_arr[inicio:inicio+TAM_LOTE]
    lote_visitorid = visitorid_arr[inicio:inicio+TAM_LOTE]

    ids_rec, _ = modelo.modelo.recommend(lote_u, mi.matriz[lote_u], N=K, filter_already_liked_items=True)

    for vid, item_real, recomendados in zip(lote_visitorid, lote_i_real, ids_rec):
        hit = 1 if item_real in list(recomendados) else 0
        resultados_por_usuario.append({"visitorid": int(vid), "hit": hit})

df_resultados = pl.DataFrame(resultados_por_usuario)
df_con_cluster = df_resultados.join(segmentos.select(["visitorid", "cluster"]), on="visitorid", how="left")

hit_rate_por_cluster = (
    df_con_cluster
    .group_by("cluster")
    .agg([pl.mean("hit").alias(f"hit_rate@{K}"), pl.len().alias("n_usuarios_evaluados")])
    .sort("cluster")
)

hit_rate_por_cluster.write_csv(CARPETA_RESULTADOS / "hit_rate_por_cluster.csv")
hit_rate_por_cluster


cluster,hit_rate@10,n_usuarios_evaluados
i32,f64,u32
0,0.028949,305780
2,0.005654,1238
3,0.023148,57369
4,0.006645,10383
5,0.013201,22801


## 7. Gestión de problemas

**Problema 1 — Cold start / usuarios con una sola interacción.** La mayoría
de los usuarios (Cluster 1 de M2, ~71% de la base) tiene una única
interacción registrada. Estos usuarios no pueden evaluarse con la estrategia
leave-one-out (no queda nada en train tras separar su único evento), por lo
que `dividir_train_test` los excluye automáticamente del conjunto de test.
Esto es una limitación conocida del filtrado colaborativo puro: para estos
usuarios haría falta un modelo basado en contenido o en popularidad como
respaldo (candidato a trabajo futuro para la Entrega 3).

**Problema 2 — Dispersión (sparsity) de la matriz usuario-producto.** Con
más de un millón de usuarios y cientos de miles de productos, la matriz de
interacciones es extremadamente dispersa, lo que puede afectar la calidad de
los factores latentes para usuarios con pocas interacciones. Se documenta
como limitación y se decide mantener `factors=64` como punto de partida
razonable, a ajustar con Optuna en la Entrega 3.

*(Agregar aquí cualquier otro obstáculo encontrado al correr esto contra el
dataset completo — tiempo de entrenamiento, memoria, etc.)*


## 8. Resumen para la sección IEEE "Modelado"

- **Modelo baseline:** ALS (Alternating Least Squares) vía la librería
  Implicit, entrenado sobre feedback implícito ponderado
  (`peso_implicito`: view=1, addtocart=3, transaction=5, definido en M1).
- **División de datos:** leave-one-out temporal (última interacción de cada
  usuario evaluable como test).
- **Métricas:** Hit Rate@10, MAP@10, NDCG@10 (ver sección 5 para los
  valores obtenidos, y `resultados/metricas_modelo_baseline.csv`).
- **Integración con M2:** el rendimiento del modelo se descompone por
  segmento de usuario, aprovechando los 6 clusters ya identificados.
- **Próximo paso (Entrega 3):** comparar este baseline contra un modelo de
  embeddings/Two-Tower en PyTorch, y optimizar hiperparámetros con Optuna.
